<h1>Import Libraries</h1>



In [3]:
# Install required packages if not available
import sys
!{sys.executable} -m pip install kagglehub pandas numpy matplotlib seaborn plotly

print("Packages installed successfully!")

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Data loading utilities
import zipfile
import os
import kagglehub

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

# Recommendation system libraries
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error

# Install implicit if not available
import subprocess
import sys

print("Checking for implicit library...")
try:
    import implicit
    print("✅ implicit library is already available!")
except ImportError:
    print("❌ implicit library not found. Installing...")
    try:
        # Try installing with pip
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'implicit'])
        print("✅ implicit library installed via pip!")
        
        # Try importing again
        import implicit
        print("✅ implicit library imported successfully!")
        
    except Exception as e:
        print(f"❌ Installation failed: {e}")
        print("💡 Please try manually installing with: pip install implicit")
        print("💡 Or try: conda install -c conda-forge implicit")
        raise ImportError("Could not install implicit library")



Packages installed successfully!
Libraries imported successfully!
Checking for implicit library...
✅ implicit library is already available!


<h1>Load the Data</h1>
<p>Takes about 1 minute</p>

In [4]:
# Download the dataset using kagglehub
dataset_path = kagglehub.dataset_download("andrewmvd/spotify-playlists")
print(f"Dataset downloaded to: {dataset_path}")

# Find the CSV file (it might be in a ZIP)
csv_files = []
zip_files = []

for file in os.listdir(dataset_path):
    file_path = os.path.join(dataset_path, file)
    if file.lower().endswith('.csv'):
        csv_files.append(file_path)
    elif file.lower().endswith('.zip'):
        zip_files.append(file_path)

print(f"Found {len(csv_files)} CSV files and {len(zip_files)} ZIP files")

# Load the data with robust error handling
df = None

if csv_files:
    # Direct CSV file found
    csv_file = csv_files[0]
    print(f"Loading CSV file: {csv_file}")
    try:
        df = pd.read_csv(
            csv_file,
            encoding='iso-8859-1',  # Permissive encoding
            sep=',',
            quotechar='"',
            escapechar='\\',
            engine='python',
            on_bad_lines='skip'
        )
    except Exception as e:
        print(f"Error loading CSV directly: {e}")

elif zip_files:
    # Extract and load from ZIP file
    zip_file = zip_files[0]
    print(f"Extracting from ZIP file: {zip_file}")
    try:
        with zipfile.ZipFile(zip_file, 'r') as z:
            csv_name = next((n for n in z.namelist() if n.lower().endswith('.csv')), None)
            if csv_name:
                print(f"Found CSV in ZIP: {csv_name}")
                with z.open(csv_name) as f:
                    df = pd.read_csv(
                        f,
                        encoding='iso-8859-1',
                        sep=',',
                        quotechar='"',
                        escapechar='\\',
                        engine='python',
                        on_bad_lines='skip'
                    )
            else:
                print("No CSV file found in ZIP archive")
    except Exception as e:
        print(f"Error extracting from ZIP: {e}")

if df is not None:
    print(f"Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
else:
    print("Failed to load dataset")

# Clean column names - remove extra spaces and quotes
if df is not None:
    print("Cleaning column names...")
    print("Before:", list(df.columns))
    
    # Strip spaces and quotes from column names
    df.columns = [col.strip().strip('"') for col in df.columns]
    
    print("After:", list(df.columns))
    print("Column names cleaned successfully!")

# Basic dataset information
print("Dataset Info:")
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumn data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

# Key statistics
stats = {
    'Total Records': df.shape[0],
    'Unique Users': df['user_id'].nunique() if 'user_id' in df.columns else 'N/A',
    'Unique Artists': df['artistname'].nunique() if 'artistname' in df.columns else 'N/A',
    'Unique Tracks': df['trackname'].nunique() if 'trackname' in df.columns else 'N/A',
    'Unique Playlists': df['playlistname'].nunique() if 'playlistname' in df.columns else 'N/A'
}

for key, value in stats.items():
    print(f"{key}: {value:,}" if isinstance(value, int) else f"{key}: {value}")

    


Dataset downloaded to: C:\Users\mstan\.cache\kagglehub\datasets\andrewmvd\spotify-playlists\versions\1
Found 1 CSV files and 0 ZIP files
Loading CSV file: C:\Users\mstan\.cache\kagglehub\datasets\andrewmvd\spotify-playlists\versions\1\spotify_dataset.csv
Dataset loaded successfully!
Shape: (12791243, 4)
Columns: ['user_id', ' "artistname"', ' "trackname"', ' "playlistname"']
Cleaning column names...
Before: ['user_id', ' "artistname"', ' "trackname"', ' "playlistname"']
After: ['user_id', 'artistname', 'trackname', 'playlistname']
Column names cleaned successfully!
Dataset Info:
Number of rows: 12,791,243
Number of columns: 4
Memory usage: 3390.16 MB

Column data types:
user_id         object
artistname      object
trackname       object
playlistname    object
dtype: object

Missing values:
user_id             0
artistname      33536
trackname          88
playlistname       41
dtype: int64
Total Records: 12,791,243
Unique Users: 15,910
Unique Artists: 287,438
Unique Tracks: 1,999,879
U

In [5]:
df

,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,"(What's So Funny 'Bout) Peace, Love And Unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,Tiffany Page,7 Years Too Late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010
...,...,...,...,...
12791238,2302bf9c64dc63d88a750215ed187f2c,MÃ¶tley CrÃ¼e,Wild Side,iPhone
12791239,2302bf9c64dc63d88a750215ed187f2c,John Lennon,Woman,iPhone
12791240,2302bf9c64dc63d88a750215ed187f2c,Tom Petty,You Don't Know How It Feels,iPhone
12791241,2302bf9c64dc63d88a750215ed187f2c,Tom Petty,You Wreck Me,iPhone


In [ ]:
# Clean and standardize artistname and trackname
import re

def clean_text(text):
    """
    Standardize text by:
    1. Converting to lowercase for consistency
    2. Standardizing common abbreviations 
    3. Removing extra punctuation and whitespace
    4. Handling special characters
    """
    if pd.isna(text) or text == '':
        return text
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Fix common encoding corruptions first
    encoding_fixes = {
        'ã¡': 'á', 'ã©': 'é', 'ã­': 'í', 'ã³': 'ó', 'ãº': 'ú',
        'ã¤': 'ä', 'ã«': 'ë', 'ã¯': 'ï', 'ã¶': 'ö', 'ã¼': 'ü',
        'ã ': 'à', 'ã¨': 'è', 'ã¬': 'ì', 'ã²': 'ò', 'ã¹': 'ù',
        'ã¢': 'â', 'ãª': 'ê', 'ã®': 'î', 'ã´': 'ô', 'ã»': 'û',
        'ã§': 'ç', 'ã±': 'ñ', 'ã¿': 'ÿ'
    }
    
    for corrupted, correct in encoding_fixes.items():
        text = text.replace(corrupted, correct)
    
    # Remove extra whitespace and normalize
    text = re.sub(r'\s+', ' ', text.strip())
    
    # Common abbreviations standardization
    abbreviations = {
        r'\bft\.?\b': 'feat',
        r'\bfeat\.?\b': 'feat', 
        r'\bfeaturing\b': 'feat',
        r'\bw\/\b': 'with',
        r'\bw\b': 'with',
        r'\s*&\s*': ' and ',  # Match & with optional spaces around it
        r'\bvs\.?\b': 'vs',
        r'\bversus\b': 'vs',
        r'\bpt\.?\b': 'part',
        r'\bvol\.?\b': 'vol',
        r'\bno\.?\b': 'no',
        r'\bst\.?\b': 'st',
        r'\bdr\.?\b': 'dr',
        r'\bmr\.?\b': 'mr',
        r'\bms\.?\b': 'ms',
    }
    
    for pattern, replacement in abbreviations.items():
        text = re.sub(pattern, replacement, text)
    
    # Remove parentheses and everything inside them
    # This consolidates "Hey Jude (Remastered)" and "Hey Jude"
    text = re.sub(r'\([^)]*\)', '', text)
    
    # Remove or standardize punctuation
    # Keep important punctuation but remove excessive
    text = re.sub(r'[""''`']', '', text)  # Remove various quote marks and apostrophes
    text = re.sub(r'[.]{2,}', '', text)  # Remove multiple dots
    text = re.sub(r'[-]{2,}', '-', text)  # Standardize multiple dashes
    text = re.sub(r'[!]{2,}', '!', text)  # Standardize multiple exclamation
    text = re.sub(r'[?]{2,}', '?', text)  # Standardize multiple questions
    text = re.sub(r'[,]{2,}', ',', text)  # Standardize multiple commas
    
    # Clean up spacing around punctuation
    text = re.sub(r'\s*,\s*', ', ', text)  # Standardize comma spacing
    text = re.sub(r'\s*-\s*', ' - ', text)  # Standardize dash spacing
    
    # Normalize accented characters to ASCII for better matching
    # This helps consolidate entries like "café" and "cafe"
    import unicodedata
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    
    # Final cleanup
    text = re.sub(r'\s+', ' ', text.strip())
    
    return text

# Show before/after examples
print("🧹 CLEANING ARTISTNAME AND TRACKNAME")
print("=" * 50)

# Sample some data to show what we're cleaning
print("Before cleaning - Sample artist names:")
sample_artists = df['artistname'].dropna().head(10).tolist()
for artist in sample_artists:
    print(f"  '{artist}'")

print("\nAfter cleaning - Same artists:")
for artist in sample_artists:
    cleaned = clean_text(artist)
    print(f"  '{artist}' → '{cleaned}'")

print(f"\nBefore cleaning - Sample track names:")
sample_tracks = df['trackname'].dropna().head(10).tolist()
for track in sample_tracks:
    print(f"  '{track}'")

print("\nAfter cleaning - Same tracks:")
for track in sample_tracks:
    cleaned = clean_text(track)
    print(f"  '{track}' → '{cleaned}'")

# Apply cleaning to the dataframe
print(f"\n🔄 Applying cleaning to full dataset...")
print(f"Original unique artists: {df['artistname'].nunique():,}")
print(f"Original unique tracks: {df['trackname'].nunique():,}")

# Create cleaned versions
df['artistname_clean'] = df['artistname'].apply(clean_text)
df['trackname_clean'] = df['trackname'].apply(clean_text)

print(f"Cleaned unique artists: {df['artistname_clean'].nunique():,}")
print(f"Cleaned unique tracks: {df['trackname_clean'].nunique():,}")

# Calculate reduction in uniqueness
artist_reduction = df['artistname'].nunique() - df['artistname_clean'].nunique()
track_reduction = df['trackname'].nunique() - df['trackname_clean'].nunique()

print(f"\n📊 CLEANING IMPACT:")
print(f"Artists consolidated: {artist_reduction:,} ({artist_reduction/df['artistname'].nunique()*100:.1f}%)")
print(f"Tracks consolidated: {track_reduction:,} ({track_reduction/df['trackname'].nunique()*100:.1f}%)")

# Replace original columns with cleaned versions
df['artistname'] = df['artistname_clean']
df['trackname'] = df['trackname_clean']

# Drop the temporary cleaned columns
df = df.drop(['artistname_clean', 'trackname_clean'], axis=1)

print(f"\n✅ Cleaning complete! Original columns updated with standardized text.")
print(f"Final dataset shape: {df.shape}")


🧹 CLEANING ARTISTNAME AND TRACKNAME
Before cleaning - Sample artist names:
  'Elvis Costello'
  'Elvis Costello & The Attractions'
  'Tiffany Page'
  'Elvis Costello & The Attractions'
  'Elvis Costello'
  'Lissie'
  'Paul McCartney'
  'Joe Echo'
  'Paul McCartney'
  'Lissie'

After cleaning - Same artists:
  'Elvis Costello' → 'elvis costello'
  'Elvis Costello & The Attractions' → 'elvis costello & the attractions'
  'Tiffany Page' → 'tiffany page'
  'Elvis Costello & The Attractions' → 'elvis costello & the attractions'
  'Elvis Costello' → 'elvis costello'
  'Lissie' → 'lissie'
  'Paul McCartney' → 'paul mccartney'
  'Joe Echo' → 'joe echo'
  'Paul McCartney' → 'paul mccartney'
  'Lissie' → 'lissie'

Before cleaning - Sample track names:
  '(The Angels Wanna Wear My) Red Shoes'
  '(What's So Funny 'Bout) Peace, Love And Understanding'
  '7 Years Too Late'
  'Accidents Will Happen'
  'Alison'
  'All Be Okay'
  'Band On The Run'
  'Beautiful'
  'Blackbird - Live at CitiField, NYC - D

,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello,(the angels wanna wear my) red shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello & the attractions,"(what's so funny 'bout) peace, love and unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,tiffany page,7 years too late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello & the attractions,accidents will happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello,alison,HARD ROCK 2010
...,...,...,...,...
12791238,2302bf9c64dc63d88a750215ed187f2c,mã¶tley crã¼e,wild side,iPhone
12791239,2302bf9c64dc63d88a750215ed187f2c,john lennon,woman,iPhone
12791240,2302bf9c64dc63d88a750215ed187f2c,tom petty,you don't know how it feels,iPhone
12791241,2302bf9c64dc63d88a750215ed187f2c,tom petty,you wreck me,iPhone


In [10]:
df.head(30)

,user_id,artistname,trackname,playlistname
0,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello,(the angels wanna wear my) red shoes,HARD ROCK 2010
1,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello & the attractions,"(what's so funny 'bout) peace, love and unders...",HARD ROCK 2010
2,9cc0cfd4d7d7885102480dd99e7a90d6,tiffany page,7 years too late,HARD ROCK 2010
3,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello & the attractions,accidents will happen,HARD ROCK 2010
4,9cc0cfd4d7d7885102480dd99e7a90d6,elvis costello,alison,HARD ROCK 2010
5,9cc0cfd4d7d7885102480dd99e7a90d6,lissie,all be okay,HARD ROCK 2010
6,9cc0cfd4d7d7885102480dd99e7a90d6,paul mccartney,band on the run,HARD ROCK 2010
7,9cc0cfd4d7d7885102480dd99e7a90d6,joe echo,beautiful,HARD ROCK 2010
8,9cc0cfd4d7d7885102480dd99e7a90d6,paul mccartney,"blackbird - live at citifield, nyc - digital a...",HARD ROCK 2010
9,9cc0cfd4d7d7885102480dd99e7a90d6,lissie,bright side,HARD ROCK 2010
